# Predicting Dengue Outbreaks from Urban Environmental Conditions in RS/BRA

In [1]:
import sys

sys.path.insert(0, '..')  # makes src/ importable from notebooks/

## 0. Problem definition

## 1. Data loading

### 1.1 Meteorological Data (INMET) 2020-2024

In [ ]:
from src.data.extract import fetch_inmet

inmet = fetch_inmet(years=range(2020, 2024))
print(inmet.shape)
inmet.head(2)

### 1.2 IBGE City Codes

In [ ]:
from src.data.extract import fetch_rs_geocodes

fetch_rs_geocodes()

### 1.3 Dengue Cases (Info.Dengue) 2020-2024

In [ ]:
from src.data.extract import fetch_dengue

dengue = fetch_dengue()
print(dengue.shape)
dengue.head(2)

### 1.4 Sanitation Data — CLEANED SNIS - SINISA 

In [ ]:
from src.data.extract import fetch_cleaned_snis_sinisa

snis_sinisa = fetch_cleaned_snis_sinisa()
print(snis_sinisa.shape)
snis_sinisa.head(2)

## 2. Data Cleaning

In [ ]:
from src.data.preprocess import (
    normalize_column_names,
    fix_mojibake,
    normalize_city_name,
    cast_br_numeric_columns,
    cast_sim_nao,
    handle_missing,
    drop_columns,
    parse_dates,
    split_categorical_numerical,
)

### 2.1 INMET

In [ ]:
inmet_clean = (
    inmet
    .pipe(normalize_column_names)
    .pipe(cast_br_numeric_columns)
    .pipe(cast_sim_nao)
    .pipe(handle_missing)
    .pipe(drop_columns, [])
    .assign(
        data=lambda df: parse_dates(df["data"], fmt="%Y/%m/%d"),
        estacao=lambda df: df["estacao"].str.strip().str.upper(),
    )
)

cat_inmet, num_inmet = split_categorical_numerical(inmet_clean)
print("Shape:", inmet_clean.shape)
print("Categorical columns:", cat_inmet.columns.tolist())
print("Numerical columns:", num_inmet.columns.tolist())
inmet_clean.head(2)

### 2.2 Dengue

In [ ]:
dengue_clean = (
    dengue
    .pipe(normalize_column_names)
    .pipe(handle_missing)
    .pipe(drop_columns, [])
    .assign(
        municipio_nome=lambda df: normalize_city_name(fix_mojibake(df["municipio_nome"])),
        data_inise=lambda df: parse_dates(df["data_inise"], fmt="%Y-%m-%d"),
    )
)

cat_dengue, num_dengue = split_categorical_numerical(dengue_clean)
print("Shape:", dengue_clean.shape)
print("Categorical columns:", cat_dengue.columns.tolist())
print("Numerical columns:", num_dengue.columns.tolist())
dengue_clean.head(2)

### 2.3 Saneamento

In [ ]:
snis_sinisa_clean = (
    snis_sinisa
    .pipe(normalize_column_names)
    .pipe(cast_br_numeric_columns)
    .pipe(cast_sim_nao)
    .pipe(handle_missing)
    .pipe(drop_columns, [])
)

cat_snis_sinisa, num_snis_sinisa = split_categorical_numerical(snis_sinisa_clean)
print("Shape:", snis_sinisa_clean.shape)
print("Categorical columns:", cat_snis_sinisa.columns.tolist())
print("Numerical columns:", num_snis_sinisa.columns.tolist())
snis_sinisa_clean.head(2)

In [ ]:
# Remover códigos numéricos dos nomes de colunas e manter apenas colunas com texto legível
snis_sinisa_clean = snis_sinisa_clean.rename(columns=lambda c: c
    .replace('_dfe0001', '')
    .replace('_dfe0002', '')
    .replace('_dfe0003', '')
    .replace('_ogm4006', '')
    .replace('_ogm4004', '')
    .replace('_ogm4005', '')
    .replace('_ogm0005', '')
    .replace('irs0001_cobertura_da_populacao_total_com_coleta_de_residuos_solidos_domiciliares_percentual', 'cobertura_total')
    .replace('irs0002_cobertura_da_populacao_urbana_com_coleta_de_residuos_solidos_domiciliares_percentual', 'cobertura_urbana')
    .replace('irs0003_cobertura_da_populacao_rural_com_coleta_de_residuos_solidos_domiciliares_percentual', 'cobertura_rural')
    .replace('irs0004_cobertura_da_populacao_urbana_com_coleta_direta_de_residuos_solidos_domiciliares_percentual', 'cobertura_urbana_direta')
    .replace('irs0005_cobertura_da_populacao_total_com_coleta_seletiva_de_residuos_solidos_domiciliares_percentual', 'coleta_seletiva_total')
    .replace('irs0006_cobertura_da_populacao_urbana_com_coleta_seletiva_direta_de_residuos_solidos_domiciliares_percentual', 'coleta_seletiva_urbana')
    .replace('municipio_nom_mun', 'municipio')
    .replace('uf', 'estado')
    .replace('codigo_do_ibge_cod_ibge', 'ibge_code')
    .replace('ano_de_referencia', 'year')
    .replace('area_km2', 'area_km2')
    .replace('populacao_total', 'populacao_total')
    .replace('populacao_urbana', 'populacao_urbana')
    .replace('populacao_rural', 'populacao_rural')
    .replace('quantidade_de_domicilios_totais_existente_no_municipio', 'domicilios_totais')
)

cat_cols_to_keep = ['municipio', 'estado']
num_cols_to_keep = [
    'ibge_code',
    'year',
    'area_km2',
    'populacao_total',
    'populacao_urbana',
    'populacao_rural',
    'domicilios_totais',
    'ids',
    'cobertura_total',
    'cobertura_urbana',
    'cobertura_rural',
    'cobertura_urbana_direta',
    'coleta_seletiva_total',
    'coleta_seletiva_urbana',
]

# Verificar quais colunas existem
existing_cols = [col for col in cat_cols_to_keep + num_cols_to_keep if col in snis_sinisa_clean.columns]

snis_sinisa_clean = snis_sinisa_clean[existing_cols]

print(f"Shape após filtro: {snis_sinisa_clean.shape}")
print(f"Colunas mantidas: {snis_sinisa_clean.columns.tolist()}")
snis_sinisa_clean.head(2)


## 3. Data Transformation

In [ ]:
from src.data.transform import (
    aggregate_inmet_weekly,
    prepare_dengue_target,
    save_transformed_data,
    load_transformed_data,
    unify_datasets
)

In [ ]:

# 3.1 Agregar INMET para semana epidemiológica
print("Agregando dados INMET...")
inmet_weekly = aggregate_inmet_weekly(inmet_clean)
print(f"Shape: {inmet_weekly.shape}")



In [ ]:
# 3.2 Preparar dados de dengue (target)
print("\nPreparando dados de dengue...")
dengue_weekly = prepare_dengue_target(dengue_clean)
print(f"Shape: {dengue_weekly.shape}")

In [ ]:
# 3.3 Verificar distribuição do target
if 'outbreak' in dengue_weekly.columns:
    print(f"\nDistribuição de surtos:")
    print(dengue_weekly['outbreak'].value_counts())
    print(f"Proporção de surtos: {dengue_weekly['outbreak'].mean():.2%}")

In [ ]:
# 3.4 Salvar dados transformados
print("\nSalvando dados transformados...")
save_transformed_data(inmet_weekly, 'inmet_weekly')
save_transformed_data(dengue_weekly, 'dengue_weekly')


In [ ]:
inmet_weekly.head(2)

In [ ]:
dengue_weekly.head(2)

## 4. Feature Engineering

In [ ]:
from src.data.features import (
    create_climate_thresholds,
    create_dengue_indices,
    create_lag_features,
    create_rolling_features,
    create_extreme_events,
    create_temporal_features,
    feature_engineering_pipeline,
    validate_features,
    print_feature_summary
)

In [ ]:
# Feature Engineering nos dados INMET

inmet_features = feature_engineering_pipeline(
    inmet_weekly,
    create_lags=True,
    create_rolling=True,
    create_extremes=True,
    create_temporal=True
)

In [ ]:
validation = validate_features(inmet_features)
print_feature_summary(validation)

In [ ]:
print("\n Salvando dataset com features...")
from src.data.transform import save_transformed_data
save_transformed_data(inmet_features, 'inmet_features')


In [ ]:
# FEATURE ENGINEERING - Saneamento

from src.data.features import (
    create_sanitation_derived_features,
    create_sanitation_binary_features,
    create_sanitation_temporal_features,
    create_sanitation_combined_features,
    sanitation_feature_engineering_pipeline,
    validate_sanitation_features,
    print_sanitation_feature_summary
)


In [ ]:

# Aplicar Feature Engineering nos dados de saneamento

snis_sinisa_features = sanitation_feature_engineering_pipeline(
    snis_sinisa_clean,
    create_derived=True,
    create_binary=True,
    create_temporal=True,
    create_combined=True
)

print(f"Shape após Feature Engineering: {snis_sinisa_features.shape}")

In [ ]:
# Validar features criadas
validation = validate_sanitation_features(snis_sinisa_features)
print_sanitation_feature_summary(validation)

In [ ]:
# Salvar dataset com features
from src.data.transform import save_transformed_data
save_transformed_data(snis_sinisa_features, 'snis_sinisa_features')

In [ ]:
# Features derivadas
derived_stats = snis_sinisa_features[[
    'densidade_populacional', 'proporcao_urbana', 'moradores_por_domicilio',
    'qualidade_servico', 'cobertura_media', 'infraestrutura_score'
]].describe() if all(c in snis_sinisa_features.columns for c in [
    'densidade_populacional', 'proporcao_urbana', 'moradores_por_domicilio',
    'qualidade_servico', 'cobertura_media', 'infraestrutura_score'
]) else None

if derived_stats is not None:
    print(derived_stats)



In [ ]:
# Distribuição das features binárias

binary_features = [
    'cobertura_alta', 'tem_coleta_seletiva', 'densidade_alta'
]

existing_binary = [f for f in binary_features if f in snis_sinisa_features.columns]

if existing_binary:
    for feat in existing_binary:
        if feat in snis_sinisa_features.columns:
            print(f"\n{feat}:")
            print(f"  True: {snis_sinisa_features[feat].sum()} ({snis_sinisa_features[feat].mean()*100:.1f}%)")
            print(f"  False: {(~snis_sinisa_features[feat].astype(bool)).sum()}")



In [ ]:
# Verificar IDS (Índice de Desenvolvimento do Saneamento)

if 'ids' in snis_sinisa_features.columns:
    print(f"  Média: {snis_sinisa_features['ids'].mean():.1f}")
    print(f"  Mediana: {snis_sinisa_features['ids'].median():.1f}")
    print(f"  Min: {snis_sinisa_features['ids'].min():.1f}")
    print(f"  Max: {snis_sinisa_features['ids'].max():.1f}")
    
    # Categorias de IDS
    ids_categories = [
        ('Muito Baixo', 0, 25),
        ('Baixo', 25, 50),
        ('Médio', 50, 70),
        ('Alto', 70, 85),
        ('Muito Alto', 85, 100)
    ]
    
    print("\n  Categorias de IDS:")
    for label, low, high in ids_categories:
        count = ((snis_sinisa_features['ids'] >= low) & (snis_sinisa_features['ids'] < high)).sum()
        pct = count / len(snis_sinisa_features) * 100
        print(f"    {label} ({low}-{high}): {count} ({pct:.1f}%)")



In [ ]:
snis_sinisa_features.head()

In [ ]:
# CRIAR DATASET UNIFICADO 

unified_dataset = unify_datasets(
    climate_df=inmet_weekly,
    dengue_df=dengue_weekly,
    sanitation_df=snis_sinisa_features,
    station_city_map=None
)


print(f"Colunas: {unified_dataset.columns.tolist()}")

print(f"\nDistribuição do target (outbreak):")
print(unified_dataset['outbreak'].value_counts())

print(f"\nFeatures numéricas: {len(unified_dataset.select_dtypes(include=['float64', 'int64']).columns)}")

print("\n4.14 Salvando dataset unificado...")
save_transformed_data(unified_dataset, 'unified_dataset')


In [ ]:

#  SEPARAR X E y PARA MODELAGEM (QUANDO FOR TREINAR)

# Separar agora OU na hora de treinar
X = unified_dataset.drop(columns=['outbreak', 'casos', 'semana_id', 'estacao'], errors='ignore')
y = unified_dataset['outbreak']

print(f"X: {X.shape}")
print(f"y: {y.shape}")

# Opcional: Salvar separadamente para modelagem
save_transformed_data(X, 'X_features')
save_transformed_data(y.to_frame(), 'y_target')

print("\n Dataset pronto!")
print(f"   - unified_dataset (com target): {unified_dataset.shape}")
print(f"   - X_features: {X.shape}")
print(f"   - y_target: {y.shape}")

## 5. Exploratory Data Analysis

## 6. Modeling

In [ ]:
### 6.1 Train/Validation/Test Split (70%/15%/15%) with Group K-Fold Cross-Validation

from sklearn.model_selection import train_test_split, GroupKFold
import numpy as np

# Identify grouping variable (use municipality if available)
group_col = None
if 'municipio_nom_mun' in unified_dataset.columns:
    group_col = 'municipio_nom_mun'
    groups = unified_dataset[group_col].values
    print(f"Using '{group_col}' as grouping variable")
    print(f"Number of groups: {unified_dataset[group_col].nunique()}")
elif 'estacao' in unified_dataset.columns:
    group_col = 'estacao'
    groups = unified_dataset[group_col].values
    print(f"Using '{group_col}' as grouping variable")
    print(f"Number of groups: {unified_dataset[group_col].nunique()}")
else:
    # Fallback: use row index modulo for grouping
    groups = np.arange(len(unified_dataset)) % 10
    print("No clear grouping variable found. Using index-based groups.")

# Check if stratification is possible (each class needs at least 2 samples)
def can_stratify(y_values):
    """Check if stratification is possible for given target values."""
    if len(np.unique(y_values)) <= 1:
        return False
    for class_label in np.unique(y_values):
        if (y_values == class_label).sum() < 2:
            return False
    return True

# Step 1: Split into train (70%) and temp (30%)
idx = np.arange(len(X))
use_stratify_1 = can_stratify(y.values)

train_idx, temp_idx = train_test_split(
    idx, 
    test_size=0.30, 
    random_state=42,
    stratify=y.values if use_stratify_1 else None
)

X_train, X_temp = X.iloc[train_idx], X.iloc[temp_idx]
y_train, y_temp = y.iloc[train_idx], y.iloc[temp_idx]
groups_train, groups_temp = groups[train_idx], groups[temp_idx]

# Step 2: Split temp into validation (50%) and test (50%) -> 15% / 15%
use_stratify_2 = can_stratify(y_temp.values)

val_idx, test_idx = train_test_split(
    np.arange(len(X_temp)),
    test_size=0.50,
    random_state=42,
    stratify=y_temp.values if use_stratify_2 else None
)

X_val = X_temp.iloc[val_idx]
X_test = X_temp.iloc[test_idx]
y_val = y_temp.iloc[val_idx]
y_test = y_temp.iloc[test_idx]

print(f"\n Train/Validation/Test Split:")
print(f"   Train:      {len(X_train):5d} samples ({len(X_train)/len(X)*100:5.1f}%)")
print(f"   Validation: {len(X_val):5d} samples ({len(X_val)/len(X)*100:5.1f}%)")
print(f"   Test:       {len(X_test):5d} samples ({len(X_test)/len(X)*100:5.1f}%)")

print(f"\n   Train positive ratio: {y_train.mean():.2%}")
print(f"   Val positive ratio:   {y_val.mean():.2%}")
print(f"   Test positive ratio:  {y_test.mean():.2%}")

if not use_stratify_1 or not use_stratify_2:
    print(f"\n  Note: Stratified split not possible due to class imbalance. Using random split.")

In [ ]:
### 6.2 Setup Group 5-Fold Cross-Validation

# Initialize GroupKFold with 5 folds
gkf = GroupKFold(n_splits=5)

# Generate fold indices using training data only
fold_indices = list(gkf.split(X_train, y_train, groups=groups_train))

print(f" Group 5-Fold Cross-Validation Setup:")
print(f"   Total folds: {len(fold_indices)}")
print(f"\n   Fold Details:")

for fold_num, (train_fold_idx, val_fold_idx) in enumerate(fold_indices, 1):
    X_fold_train = X_train.iloc[train_fold_idx]
    X_fold_val = X_train.iloc[val_fold_idx]
    y_fold_train = y_train.iloc[train_fold_idx]
    y_fold_val = y_train.iloc[val_fold_idx]
    
    print(f"\n   Fold {fold_num}:")
    print(f"      Train: {len(X_fold_train)} samples | Positive: {y_fold_train.mean():.2%}")
    print(f"      Val:   {len(X_fold_val)} samples   | Positive: {y_fold_val.mean():.2%}")

# Save fold indices for use in training
fold_info = {
    'fold_indices': fold_indices,
    'train_idx': train_idx,
    'val_idx': temp_idx[val_idx],
    'test_idx': temp_idx[test_idx],
    'groups_train': groups_train
}

print(f"\n Fold information saved for cross-validation training.")

In [ ]:
### 6.3 Save Split Data for Model Training

# Save all splits for reproducibility
save_transformed_data(X_train, 'X_train_split')
save_transformed_data(X_val, 'X_val_split')
save_transformed_data(X_test, 'X_test_split')
save_transformed_data(y_train.to_frame(), 'y_train_split')
save_transformed_data(y_val.to_frame(), 'y_val_split')
save_transformed_data(y_test.to_frame(), 'y_test_split')

print(" Split datasets saved:")
print(f"   X_train_split: {X_train.shape}")
print(f"   X_val_split:   {X_val.shape}")
print(f"   X_test_split:  {X_test.shape}")
print(f"\n   y_train_split: {y_train.shape}")
print(f"   y_val_split:   {y_val.shape}")
print(f"   y_test_split:  {y_test.shape}")

# Summary statistics
print(f"\n Data Split Summary:")
print(f"   Original dataset: {X.shape[0]} samples")
print(f"   Train (70%):      {X_train.shape[0]} samples")
print(f"   Validation (15%): {X_val.shape[0]} samples")
print(f"   Test (15%):       {X_test.shape[0]} samples")
print(f"\n   Cross-Validation: Group 5-Fold (on training set)")
print(f"   Grouping variable: {group_col if group_col else 'Index-based'}")
print(f"\n Ready for model training with cross-validation!")

### 6.4 XGBoost Model Training & Evaluation

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc
)
import matplotlib.pyplot as plt
import numpy as np

# Initialize XGBoost Classifier
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)

# Train on training set
print(" Training XGBoost Model...")
xgb_model.fit(X_train, y_train)
print(" Model trained successfully!")


In [ ]:
# Evaluate on Validation Set
print("\n" + "="*60)
print(" VALIDATION SET EVALUATION")
print("="*60)

y_val_pred = xgb_model.predict(X_val)
y_val_pred_proba = xgb_model.predict_proba(X_val)[:, 1]

val_accuracy = accuracy_score(y_val, y_val_pred)
val_precision = precision_score(y_val, y_val_pred, zero_division=0)
val_recall = recall_score(y_val, y_val_pred, zero_division=0)
val_f1 = f1_score(y_val, y_val_pred, zero_division=0)
val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)

print(f"\n Metrics:")
print(f"   Accuracy:  {val_accuracy:.4f}")
print(f"   Precision: {val_precision:.4f}")
print(f"   Recall:    {val_recall:.4f}")
print(f"   F1-Score:  {val_f1:.4f}")
print(f"   ROC-AUC:   {val_roc_auc:.4f}")

print(f"\n Confusion Matrix:")
val_cm = confusion_matrix(y_val, y_val_pred)
print(f"   TN: {val_cm[0, 0]}, FP: {val_cm[0, 1]}")
print(f"   FN: {val_cm[1, 0]}, TP: {val_cm[1, 1]}")

print(f"\n Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=['No Outbreak', 'Outbreak']))


In [ ]:
# Evaluate on Test Set
print("\n" + "="*60)
print(" TEST SET EVALUATION")
print("="*60)

y_test_pred = xgb_model.predict(X_test)
y_test_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, y_test_pred_proba)

print(f"\n Metrics:")
print(f"   Accuracy:  {test_accuracy:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")
print(f"   ROC-AUC:   {test_roc_auc:.4f}")

print(f"\n Confusion Matrix:")
test_cm = confusion_matrix(y_test, y_test_pred)
print(f"   TN: {test_cm[0, 0]}, FP: {test_cm[0, 1]}")
print(f"   FN: {test_cm[1, 0]}, TP: {test_cm[1, 1]}")

print(f"\n Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=['No Outbreak', 'Outbreak']))


In [ ]:
# Comparison Summary
print("\n" + "="*60)
print("TRAIN vs VALIDATION vs TEST COMPARISON")
print("="*60)

y_train_pred = xgb_model.predict(X_train)
y_train_pred_proba = xgb_model.predict_proba(X_train)[:, 1]

train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
train_roc_auc = roc_auc_score(y_train, y_train_pred_proba)

comparison_data = {
    'Dataset': ['Train', 'Validation', 'Test'],
    'Accuracy': [train_accuracy, val_accuracy, test_accuracy],
    'Precision': [
        precision_score(y_train, y_train_pred, zero_division=0),
        val_precision,
        test_precision
    ],
    'Recall': [
        recall_score(y_train, y_train_pred, zero_division=0),
        val_recall,
        test_recall
    ],
    'F1-Score': [train_f1, val_f1, test_f1],
    'ROC-AUC': [train_roc_auc, val_roc_auc, test_roc_auc]
}

import pandas as pd
comparison_df = pd.DataFrame(comparison_data)
print("\n")
print(comparison_df.to_string(index=False))

# Check for overfitting
print("\n Overfitting Check:")
overfit_gap = train_accuracy - test_accuracy
print(f"   Accuracy gap (Train - Test): {overfit_gap:.4f}")
if overfit_gap > 0.10:
    print(f"  Possible overfitting detected (gap > 0.10)")
else:
    print(f"  No significant overfitting")


In [ ]:
# Feature Importance
print("\n" + "="*60)
print("🔍 TOP 15 MOST IMPORTANT FEATURES")
print("="*60)

feature_importance = xgb_model.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\n")
for idx, row in importance_df.head(15).iterrows():
    bar_length = int(row['Importance'] * 50)
    bar = '█' * bar_length
    print(f"{row['Feature']:30s} {bar} {row['Importance']:.4f}")

# Visualize top 10 features
plt.figure(figsize=(10, 6))
top_10_importance = importance_df.head(10)
plt.barh(range(len(top_10_importance)), top_10_importance['Importance'])
plt.yticks(range(len(top_10_importance)), top_10_importance['Feature'])
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Important Features (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n Feature importance visualization displayed.")


In [ ]:
# ROC Curves Comparison
plt.figure(figsize=(10, 8))

# Calculate ROC curves
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred_proba)
fpr_val, tpr_val, _ = roc_curve(y_val, y_val_pred_proba)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_proba)

# Plot ROC curves
plt.plot(fpr_train, tpr_train, label=f'Train (AUC = {train_roc_auc:.3f})', linewidth=2)
plt.plot(fpr_val, tpr_val, label=f'Validation (AUC = {val_roc_auc:.3f})', linewidth=2)
plt.plot(fpr_test, tpr_test, label=f'Test (AUC = {test_roc_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - XGBoost Model Performance', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(" ROC curves displayed.")


In [ ]:
# Save Model and Results
import pickle
from pathlib import Path

model_dir = Path('../data/processed/models')
model_dir.mkdir(parents=True, exist_ok=True)

# Save the trained model
model_path = model_dir / 'xgboost_dengue_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(xgb_model, f)
print(f" Model saved to: {model_path}")

# Save predictions
pred_results = pd.DataFrame({
    'y_test': y_test.values,
    'y_test_pred': y_test_pred,
    'y_test_pred_proba': y_test_pred_proba
})
pred_results_path = model_dir / 'xgboost_test_predictions.csv'
pred_results.to_csv(pred_results_path, index=False)
print(f" Test predictions saved to: {pred_results_path}")

# Summary Report
print("\n" + "="*60)
print(" FINAL SUMMARY - XGBOOST MODEL")
print("="*60)
print(f"\n Model Configuration:")
print(f"   - Estimators: 100")
print(f"   - Max Depth: 6")
print(f"   - Learning Rate: 0.1")
print(f"   - Subsample: 0.8")
print(f"   - Colsample_bytree: 0.8")

print(f"\n Test Set Performance:")
print(f"   - Accuracy:  {test_accuracy:.4f}")
print(f"   - Precision: {test_precision:.4f}")
print(f"   - Recall:    {test_recall:.4f}")
print(f"   - F1-Score:  {test_f1:.4f}")
print(f"   - ROC-AUC:   {test_roc_auc:.4f}")

print(f"\n Top Feature: {importance_df.iloc[0]['Feature']} (Importance: {importance_df.iloc[0]['Importance']:.4f})")
print(f"\n✨ Ready for deployment!")


## 7. Evaluation

## 8. Results & Discussion

## 9. Limitations & Future Work